In [ ]:
# from pathlib import Path

# file = Path.home() / "Downloads"

# for f in file.iterdir():
#     print(f, f.suffix)




In [22]:
from pathlib import Path
import pandas as pd

# ==========================
# PATHS
# ==========================
OUT = Path("flight_data")
OUT.mkdir(exist_ok=True)

DATA_DIR = Path.home() / "Downloads"
FILE_PATH = DATA_DIR / "On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv"

output_file = OUT / "airport_hourly_flights_2025_001.csv"

# ==========================
# AIRPORTS
# ==========================
AIRPORTS = [
    "TPA","STL","SNA","SMF","SLC","SJC","SFO","SEA","SAT","SAN","RSW","RDU","PIT",
    "PHX","PHL","PDX","ORD","OGG","OAK","MSY","MSP","MIA","MDW","MCO","MCI","LGA",
    "LAX","LAS","JFK","IND","IAH","IAD","HOU","HNL","FLL","EWR","DTW","DFW","DEN",
    "DCA","DAL","CVG","CMH","CLT","CLE","BWI","BOS","BNA","AUS","ATL"
]

# ==========================
# LOAD
# ==========================
def load_data():
    print(f"Loading: {FILE_PATH}")

    if not FILE_PATH.exists():
        raise FileNotFoundError(f"File not found: {FILE_PATH}")

    df = pd.read_csv(FILE_PATH, low_memory=False)

    # clean column names
    df.columns = df.columns.str.strip()

    return df

# ==========================
# PROCESS
# ==========================
def process(df):

    # ==========================
    # CLEAN TIMES (ACTUAL TIMES)
    # ==========================
    df["DepTime"] = pd.to_numeric(df["DepTime"], errors="coerce")
    df["ArrTime"] = pd.to_numeric(df["ArrTime"], errors="coerce")

    # remove cancelled / invalid rows
    df = df.dropna(subset=["DepTime", "ArrTime", "FlightDate"])

    df["DepTime"] = df["DepTime"].astype(int)
    df["ArrTime"] = df["ArrTime"].astype(int)

    # ==========================
    # HANDLE OVERNIGHT FLIGHTS
    # ==========================
    df["FlightDate"] = pd.to_datetime(df["FlightDate"])

    overnight_mask = df["ArrTime"] < df["DepTime"]
    df.loc[overnight_mask, "FlightDate"] += pd.Timedelta(days=1)

    # ==========================
    # EXTRACT HOURS
    # ==========================
    df["DepHour"] = df["DepTime"] // 100
    df["ArrHour"] = df["ArrTime"] // 100

    # ==========================
    # FILTER AIRPORTS
    # ==========================
    df = df[
        (df["Origin"].isin(AIRPORTS)) |
        (df["Dest"].isin(AIRPORTS))
    ].copy()

    # ==========================
    # DEPARTURES
    # ==========================
    dep = (
        df[df["Origin"].isin(AIRPORTS)]
        .groupby(["FlightDate", "Origin", "DepHour"])
        .size()
        .reset_index(name="Departures")
        .rename(columns={
            "Origin": "Airport",
            "DepHour": "Hour"
        })
    )

    # ==========================
    # ARRIVALS
    # ==========================
    arr = (
        df[df["Dest"].isin(AIRPORTS)]
        .groupby(["FlightDate", "Dest", "ArrHour"])
        .size()
        .reset_index(name="Arrivals")
        .rename(columns={
            "Dest": "Airport",
            "ArrHour": "Hour"
        })
    )

    # ==========================
    # MERGE
    # ==========================
    merged = pd.merge(
        dep,
        arr,
        on=["FlightDate", "Airport", "Hour"],
        how="outer"
    ).fillna(0)

    merged["Departures"] = merged["Departures"].astype(int)
    merged["Arrivals"] = merged["Arrivals"].astype(int)

    merged["TotalFlights"] = merged["Departures"] + merged["Arrivals"]

    # sort nicely
    merged = merged.sort_values(["FlightDate", "Airport", "Hour"])

    return merged

# ==========================
# RUN
# ==========================
df = load_data()
out = process(df)

out.to_csv(output_file, index=False)

print("Saved:", output_file)
print(out.head())

Loading: C:\Users\alecg\Downloads\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv
Saved: flight_data\airport_hourly_flights_2025_001.csv
  FlightDate Airport  Hour  Departures  Arrivals  TotalFlights
0 2025-01-01     ATL     0           1         0             1
1 2025-01-01     ATL     5           5         1             6
2 2025-01-01     ATL     6          11         4            15
3 2025-01-01     ATL     7          23        38            61
4 2025-01-01     ATL     8          56        66           122
